# Sprint 1 — Data Pipeline
## IPL Real-Time Match Decision Support Dashboard
Merges all 175 CSV files across 11 stat categories (2008–2022) into a clean master dataset, engineers features, and exports ready-to-use DataFrames for ML modelling.

In [4]:
#Install dependencies (run once)
!pip install pandas numpy matplotlib seaborn

In [5]:
import pandas as pd
import numpy as np
import glob
import os
import warnings
warnings.filterwarnings('ignore')

# ── Set this to your dataset root folder ──
BASE_DIR = "IPL - Player Performance Dataset"

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Dataset root:", BASE_DIR)

Pandas: 3.0.2
NumPy: 2.4.4
Dataset root: IPL - Player Performance Dataset


## 1. Utility Functions

In [6]:
def safe_float(val):
    """Convert messy IPL CSV values to float safely."""
    try:
        return float(str(val).replace('*', '').replace('-', '0').strip() or 0)
    except:
        return 0.0

def safe_int(val):
    """Convert messy IPL CSV values to int safely."""
    try:
        return int(str(val).replace('*', '').replace('-', '0').strip() or 0)
    except:
        return 0

def extract_year(filepath):
    """Extract season year from filename like 'Most Runs - 2022.csv'."""
    import re
    match = re.search(r'(\d{4})', os.path.basename(filepath))
    return int(match.group(1)) if match else None

print("Utility functions ready.")

Utility functions ready.


## 2. Load Batting Data — Most Runs (Season Totals)

In [7]:
batting_frames = []

for filepath in sorted(glob.glob(f"{BASE_DIR}/Most Runs/*.csv")):
    year = extract_year(filepath)
    if not year:
        continue
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    df['season'] = year
    df['source'] = 'most_runs'
    batting_frames.append(df)

batting_df = pd.concat(batting_frames, ignore_index=True)

# Standardise columns
batting_df = batting_df.rename(columns={
    'Player': 'player', 'Mat': 'matches', 'Inns': 'innings',
    'NO': 'not_out', 'Runs': 'runs', 'HS': 'highest_score',
    'Avg': 'bat_avg', 'BF': 'balls_faced', 'SR': 'bat_sr',
    '100': 'centuries', '50': 'fifties', '4s': 'fours', '6s': 'sixes'
})

for col in ['bat_avg', 'bat_sr', 'runs', 'balls_faced', 'fours', 'sixes', 'centuries', 'fifties']:
    batting_df[col] = batting_df[col].apply(safe_float)

batting_df['player'] = batting_df['player'].str.strip()
batting_df = batting_df.dropna(subset=['player'])
batting_df = batting_df[batting_df['player'] != '']

print(f"Batting records loaded: {len(batting_df)}")
print(f"Seasons covered: {sorted(batting_df['season'].unique())}")
batting_df[['player','season','runs','bat_avg','bat_sr','fours','sixes']].head(8)

Batting records loaded: 2148
Seasons covered: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


,player,season,runs,bat_avg,bat_sr,fours,sixes
0,Shaun Marsh,2008,616.0,68.44,139.68,59.0,26.0
1,Gautam Gambhir,2008,534.0,41.07,140.89,68.0,8.0
2,Sanath Jayasuriya,2008,518.0,43.16,167.63,58.0,31.0
3,Shane Watson,2008,472.0,47.20,151.76,47.0,19.0
4,Graeme Smith,2008,441.0,49.00,121.82,54.0,8.0
5,Adam Gilchrist,2008,436.0,33.53,137.10,51.0,19.0
6,Yusuf Pathan,2008,435.0,31.07,179.01,43.0,25.0
7,Suresh Raina,2008,421.0,38.27,142.22,35.0,18.0


## 3. Load Bowling Data — Most Wickets (Season Totals)

In [8]:
bowling_frames = []

for filepath in sorted(glob.glob(f"{BASE_DIR}/Most Wickets/*.csv")):
    year = extract_year(filepath)
    if not year:
        continue
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    df['season'] = year
    df['source'] = 'most_wickets'
    bowling_frames.append(df)

bowling_df = pd.concat(bowling_frames, ignore_index=True)

bowling_df = bowling_df.rename(columns={
    'Player': 'player', 'Mat': 'matches', 'Inns': 'innings',
    'Ov': 'overs', 'Runs': 'runs_conceded', 'Wkts': 'wickets',
    'BBI': 'best_bowling', 'Avg': 'bowl_avg', 'Econ': 'economy',
    'SR': 'bowl_sr', '4w': 'four_wickets', '5w': 'five_wickets'
})

for col in ['wickets', 'economy', 'bowl_avg', 'bowl_sr', 'overs', 'runs_conceded']:
    bowling_df[col] = bowling_df[col].apply(safe_float)

bowling_df['player'] = bowling_df['player'].str.strip()
bowling_df = bowling_df.dropna(subset=['player'])

print(f"Bowling records loaded: {len(bowling_df)}")
bowling_df[['player','season','wickets','economy','bowl_avg','bowl_sr']].head(8)

Bowling records loaded: 1631


,player,season,wickets,economy,bowl_avg,bowl_sr
0,Sohail Tanvir,2008,22.0,6.46,12.09,11.22
1,Shane Warne,2008,19.0,7.76,21.26,16.42
2,Shanthakumaran Sreesanth,2008,19.0,8.63,23.26,16.15
3,Shane Watson,2008,17.0,7.07,22.52,19.11
4,Manpreet Gony,2008,17.0,7.38,26.05,21.17
5,Piyush Chawla,2008,17.0,8.30,22.88,16.52
6,Albie Morkel,2008,17.0,8.31,23.47,16.94
7,Yo Mahesh,2008,16.0,8.77,23.12,15.81


## 4. Load Innings-Level Data (Economy, Dot Balls, Strike Rate, Runs Conceded)

In [9]:
def load_innings_category(pattern, rename_map, numeric_cols):
    """Generic loader for innings-level bowling CSVs."""
    frames = []
    for filepath in sorted(glob.glob(pattern)):
        year = extract_year(filepath)
        if not year:
            continue
        df = pd.read_csv(filepath)
        df.columns = df.columns.str.strip()
        df['season'] = year
        frames.append(df)

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    combined = combined.rename(columns=rename_map)
    for col in numeric_cols:
        if col in combined.columns:
            combined[col] = combined[col].apply(safe_float)
    combined['player'] = combined['player'].str.strip()
    return combined.dropna(subset=['player'])

# Economy innings
economy_inn = load_innings_category(
    f"{BASE_DIR}/Best Bowling Economy Innings/*.csv",
    {'Player':'player','Ov':'overs_bowled','Runs':'runs_given','Wkts':'wickets_inn',
     'Dots':'dots','Econ':'economy_inn','SR':'sr_inn','Against':'opponent','Venue':'venue','Match Date':'match_date'},
    ['overs_bowled','runs_given','wickets_inn','dots','economy_inn','sr_inn']
)

# Dot balls innings
dots_inn = load_innings_category(
    f"{BASE_DIR}/Most Dot Balls Innings/*.csv",
    {'Player':'player','Ov':'overs_bowled','Runs':'runs_given','Wkts':'wickets_inn',
     'Dots':'dots','SR':'sr_inn','Against':'opponent','Venue':'venue','Match Date':'match_date'},
    ['overs_bowled','runs_given','wickets_inn','dots','sr_inn']
)

# Runs conceded innings
conceded_inn = load_innings_category(
    f"{BASE_DIR}/Most Runs Conceded Innings/*.csv",
    {'Player':'player','Ov':'overs_bowled','Runs':'runs_given','Wkts':'wickets_inn',
     'SR':'sr_inn','Against':'opponent','Venue':'venue','Match Date':'match_date'},
    ['overs_bowled','runs_given','wickets_inn','sr_inn']
)

# Bowl SR innings
bowl_sr_inn = load_innings_category(
    f"{BASE_DIR}/Best Bowling Strike Rate Innings/*.csv",
    {'Player':'player','Ov':'overs_bowled','Runs':'runs_given','Wkts':'wickets_inn',
     'SR':'sr_inn','Against':'opponent','Venue':'venue','Match Date':'match_date'},
    ['overs_bowled','runs_given','wickets_inn','sr_inn']
)

print(f"Economy innings: {len(economy_inn)} records")
print(f"Dot ball innings: {len(dots_inn)} records")
print(f"Runs conceded innings: {len(conceded_inn)} records")
print(f"Bowl SR innings: {len(bowl_sr_inn)} records")

Economy innings: 2624 records
Dot ball innings: 2208 records
Runs conceded innings: 2200 records
Bowl SR innings: 2148 records


## 5. Load Batting Innings-Level Data

In [10]:
def load_batting_innings(pattern, rename_map, numeric_cols):
    frames = []
    for filepath in sorted(glob.glob(pattern)):
        year = extract_year(filepath)
        if not year:
            continue
        df = pd.read_csv(filepath)
        df.columns = df.columns.str.strip()
        df['season'] = year
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    combined = pd.concat(frames, ignore_index=True)
    combined = combined.rename(columns=rename_map)
    for col in numeric_cols:
        if col in combined.columns:
            combined[col] = combined[col].apply(safe_float)
    combined['player'] = combined['player'].str.strip()
    return combined.dropna(subset=['player'])

bat_inn_map = {'Player':'player','Runs':'runs_inn','BF':'balls_inn',
               'SR':'sr_inn','4s':'fours_inn','6s':'sixes_inn',
               'Against':'opponent','Venue':'venue','Match Date':'match_date'}
bat_num = ['runs_inn','balls_inn','sr_inn','fours_inn','sixes_inn']

fours_inn  = load_batting_innings(f"{BASE_DIR}/Most Fours Innings/*.csv",   bat_inn_map, bat_num)
sixes_inn  = load_batting_innings(f"{BASE_DIR}/Most Sixes Innings/*.csv",   bat_inn_map, bat_num)
fast_50    = load_batting_innings(f"{BASE_DIR}/Fastest Fifties/*.csv",       bat_inn_map, bat_num)
fast_100   = load_batting_innings(f"{BASE_DIR}/Fastest Centuries/*.csv",     bat_inn_map, bat_num)
runs_over  = load_batting_innings(f"{BASE_DIR}/Most Runs Over/*.csv",        bat_inn_map, bat_num)

print(f"Most Fours innings:    {len(fours_inn)}")
print(f"Most Sixes innings:    {len(sixes_inn)}")
print(f"Fastest Fifties:       {len(fast_50)}")
print(f"Fastest Centuries:     {len(fast_100)}")
print(f"Most Runs in an Over:  {len(runs_over)}")

Most Fours innings:    2786
Most Sixes innings:    2614
Fastest Fifties:       1484
Fastest Centuries:     74
Most Runs in an Over:  2100


## 6. Build Master Player Profile

In [11]:
# ── Batting season aggregates ──
bat_agg = batting_df.groupby('player').agg(
    total_runs        = ('runs',      'sum'),
    bat_avg           = ('bat_avg',   'mean'),
    bat_sr            = ('bat_sr',    'mean'),
    total_fours       = ('fours',     'sum'),
    total_sixes       = ('sixes',     'sum'),
    centuries         = ('centuries', 'sum'),
    fifties           = ('fifties',   'sum'),
    seasons_batted    = ('season',    'nunique'),
    last_season_bat   = ('season',    'max')
).reset_index()

# ── Bowling season aggregates ──
bowl_agg = bowling_df.groupby('player').agg(
    total_wickets     = ('wickets',       'sum'),
    bowl_avg          = ('bowl_avg',      'mean'),
    economy           = ('economy',       'mean'),
    bowl_sr           = ('bowl_sr',       'mean'),
    seasons_bowled    = ('season',        'nunique'),
    last_season_bowl  = ('season',        'max')
).reset_index()

# ── Innings-level bowling best economy ──
econ_best = economy_inn.groupby('player').agg(
    best_economy      = ('economy_inn',   'min'),
    avg_dots_per_spell= ('dots',          'mean'),
    inn_wickets       = ('wickets_inn',   'sum')
).reset_index()

# ── Batting innings aggreagate ──
bat_inn_combined = pd.concat([fours_inn, sixes_inn, fast_50, fast_100, runs_over], ignore_index=True)
bat_inn_agg = bat_inn_combined.groupby('player').agg(
    max_sr_inn  = ('sr_inn',    'max'),
    avg_sr_inn  = ('sr_inn',    'mean'),
    max_6s_inn  = ('sixes_inn', 'max'),
    max_4s_inn  = ('fours_inn', 'max')
).reset_index()

# ── Merge everything ──
master = bat_agg.merge(bowl_agg,   on='player', how='outer')
master = master.merge(econ_best,   on='player', how='left')
master = master.merge(bat_inn_agg, on='player', how='left')

# Fill NaN
for col in master.select_dtypes(include='number').columns:
    master[col] = master[col].fillna(0)

# Assign role
def assign_role(row):
    has_bat  = row['total_runs'] > 100
    has_bowl = row['total_wickets'] > 5 or row['economy'] > 0
    if has_bat and has_bowl: return 'allrounder'
    if has_bowl:             return 'bowler'
    return 'batter'

master['role'] = master.apply(assign_role, axis=1)

print(f"Master player profiles: {len(master)}")
print(f"Role distribution:\n{master['role'].value_counts()}")
master.head()

Master player profiles: 652
Role distribution:
role
bowler        293
batter        191
allrounder    168
Name: count, dtype: int64


,player,total_runs,bat_avg,bat_sr,total_fours,total_sixes,centuries,fifties,seasons_batted,last_season_bat,...,seasons_bowled,last_season_bowl,best_economy,avg_dots_per_spell,inn_wickets,max_sr_inn,avg_sr_inn,max_6s_inn,max_4s_inn,role
0,AB de Villiers,4697.0,37.075385,147.285385,374.0,239.0,2.0,37.0,13.0,2021.0,...,0.0,0.0,0.0,0.0,0.0,533.33,259.184200,12.0,19.0,batter
1,Aakash Chopra,53.0,9.700000,69.325000,7.0,0.0,0.0,0.0,2.0,2009.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,batter
2,Aaron Finch,2091.0,22.634545,119.692727,214.0,78.0,0.0,15.0,11.0,2022.0,...,3.0,2013.0,0.0,0.0,0.0,450.00,212.466207,6.0,12.0,allrounder
3,Aavishkar Salvi,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,2011.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,bowler
4,Abdul Samad,226.0,12.176667,118.493333,12.0,14.0,0.0,0.0,3.0,2022.0,...,2.0,2021.0,0.0,0.0,0.0,375.00,268.550000,3.0,2.0,allrounder


## 7. Feature Engineering

In [12]:
# ── Normalised Impact Score (0–100) ──
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 100))

bat_score_cols = ['total_runs', 'bat_avg', 'bat_sr', 'centuries', 'fifties', 'total_sixes']
bowl_score_cols = ['total_wickets', 'best_economy', 'avg_dots_per_spell', 'inn_wickets']

# fill 0 for missing
for col in bat_score_cols + bowl_score_cols:
    if col not in master.columns:
        master[col] = 0

master_scaled = master.copy()
master_scaled[bat_score_cols] = scaler.fit_transform(master[bat_score_cols].fillna(0))
master_scaled[bowl_score_cols] = scaler.fit_transform(master[bowl_score_cols].fillna(0))

# Weighted composite impact
master['bat_impact'] = (
    master_scaled['total_runs']   * 0.25 +
    master_scaled['bat_avg']      * 0.20 +
    master_scaled['bat_sr']       * 0.20 +
    master_scaled['centuries']    * 0.15 +
    master_scaled['fifties']      * 0.10 +
    master_scaled['total_sixes']  * 0.10
)

master['bowl_impact'] = (
    master_scaled['total_wickets']      * 0.35 +
    master_scaled['best_economy']       * 0.30 +
    master_scaled['avg_dots_per_spell'] * 0.20 +
    master_scaled['inn_wickets']        * 0.15
)

# Overall impact (role-weighted)
master['overall_impact'] = master.apply(lambda r:
    r['bat_impact']  if r['role'] == 'batter'  else
    r['bowl_impact'] if r['role'] == 'bowler'  else
    (r['bat_impact'] + r['bowl_impact']) / 2,
    axis=1
)

# Phase suitability flags
master['powerplay_bowler'] = (master['economy'] < 7.0) & (master['total_wickets'] > 5)
master['death_bowler']     = (master['economy'] < 9.0) & (master['best_economy'] < 7.0)
master['finisher']         = (master['bat_sr'] > 140)  & (master['total_sixes'] > 15)
master['anchor']           = (master['bat_avg'] > 35)  & (master['bat_sr'] < 140)

print("Feature engineering complete.")
print(master[['player','role','bat_impact','bowl_impact','overall_impact']].sort_values('overall_impact',ascending=False).head(10))

Feature engineering complete.
               player        role  bat_impact  bowl_impact  overall_impact
304    Lasith Malinga      bowler    5.473072    64.507731       64.507731
242    Jasprit Bumrah      bowler    6.601904    60.765083       60.765083
650  Yuzvendra Chahal      bowler    3.361399    60.244709       60.244709
0      AB de Villiers      batter   56.866220     0.000000       56.866220
318          MS Dhoni      batter   51.435655     0.000000       51.435655
503    Sandeep Sharma      bowler    5.622659    51.244352       51.244352
258       Jos Buttler      batter   51.009128     0.000000       51.009128
363    Mohammad Shami      bowler    5.800003    47.920440       47.920440
266          KL Rahul      batter   47.392047     0.000000       47.392047
524      Shane Watson  allrounder   51.910407    42.577522       47.243964


## 8. Venue & Opponent Aggregates

In [13]:
# ── Venue-level bowling economy ──
if 'venue' in economy_inn.columns:
    venue_bowl = economy_inn.groupby(['player','venue']).agg(
        venue_economy = ('economy_inn', 'mean'),
        venue_wkts    = ('wickets_inn', 'sum')
    ).reset_index()
    print("Venue bowling profiles:", len(venue_bowl))
    print(venue_bowl.head())

# ── Opponent-level batting SR ──
if 'opponent' in fours_inn.columns:
    bat_combined = pd.concat([fours_inn, sixes_inn], ignore_index=True)
    opp_bat = bat_combined.groupby(['player','opponent']).agg(
        opp_sr   = ('sr_inn',    'mean'),
        opp_runs = ('runs_inn',  'sum'),
        opp_6s   = ('sixes_inn', 'sum')
    ).reset_index()
    print("\nOpponent batting profiles:", len(opp_bat))
    print(opp_bat.head())

Venue bowling profiles: 1508
                  player                               venue  venue_economy  \
0  Abhishek Jhunjhunwala                         Chidambaram           4.33   
1         Abhishek Nayar  Rajiv Gandhi Intl. Cricket Stadium           6.00   
2         Abhishek Nayar                    St George's Park           4.33   
3        Abhishek Sharma                         Chidambaram           6.00   
4        Abhishek Sharma             Sharjah Cricket Stadium           4.50   

   venue_wkts  
0         1.0  
1         1.0  
2         3.0  
3         2.0  
4         0.0  

Opponent batting profiles: 1660
           player opponent      opp_sr  opp_runs  opp_6s
0  AB de Villiers      CSK  173.157143     325.0    18.0
1  AB de Villiers       DC  186.335455     682.0    37.0
2  AB de Villiers      DEC  276.470000      94.0     6.0
3  AB de Villiers       GL  208.075000     416.0    34.0
4  AB de Villiers      KKR  207.052000     614.0    39.0


## 9. Rolling Form Index (Last 3 Seasons)

In [14]:
# Batting rolling form
bat_season = batting_df[['player','season','runs','bat_avg','bat_sr']].copy()
bat_season = bat_season.sort_values(['player','season'])
bat_season['runs_3yr_avg']  = bat_season.groupby('player')['runs'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
bat_season['sr_3yr_avg'] = bat_season.groupby('player')['bat_sr'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

# Bowling rolling form
bowl_season = bowling_df[['player','season','wickets','economy']].copy()
bowl_season = bowl_season.sort_values(['player','season'])
bowl_season['wkts_3yr_avg'] = bowl_season.groupby('player')['wickets'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
bowl_season['econ_3yr_avg'] = bowl_season.groupby('player')['economy'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

# Get most recent form
bat_form  = bat_season.groupby('player').last()[['runs_3yr_avg','sr_3yr_avg']].reset_index()
bowl_form = bowl_season.groupby('player').last()[['wkts_3yr_avg','econ_3yr_avg']].reset_index()

master = master.merge(bat_form,  on='player', how='left')
master = master.merge(bowl_form, on='player', how='left')
master[['runs_3yr_avg','sr_3yr_avg','wkts_3yr_avg','econ_3yr_avg']] =     master[['runs_3yr_avg','sr_3yr_avg','wkts_3yr_avg','econ_3yr_avg']].fillna(0)

print("Rolling form features added.")
master[['player','runs_3yr_avg','sr_3yr_avg','wkts_3yr_avg','econ_3yr_avg']].sort_values('runs_3yr_avg',ascending=False).head(10)

Rolling form features added.


,player,runs_3yr_avg,sr_3yr_avg,wkts_3yr_avg,econ_3yr_avg
266,KL Rahul,629.666667,134.506667,0.0,0.00
263,K L Rahul,616.000000,135.380000,0.0,0.00
534,Shikhar Dhawan,555.000000,130.670000,2.0,7.33
434,Quinton De Kock,508.000000,148.970000,0.0,0.00
183,Faf du Plessis,492.666667,134.103333,0.0,16.00
258,Jos Buttler,481.666667,148.850000,0.0,0.00
182,Faf Du Plessis,468.000000,127.520000,0.0,0.00
550,Shubman Gill,467.000000,123.060000,0.0,0.00
435,Quinton de Kock,443.000000,129.806667,0.0,0.00
506,Sanju Samson,439.000000,147.466667,0.0,0.00


## 10. Export Clean Datasets

In [15]:
import os
os.makedirs("outputs", exist_ok=True)

# Master player profiles
master.to_csv("outputs/master_players.csv", index=False)
print(f"Saved: outputs/master_players.csv ({len(master)} rows, {len(master.columns)} cols)")

# Season-level batting & bowling (for ML training)
batting_df.to_csv("outputs/batting_seasons.csv", index=False)
bowling_df.to_csv("outputs/bowling_seasons.csv", index=False)
print(f"Saved: outputs/batting_seasons.csv  ({len(batting_df)} rows)")
print(f"Saved: outputs/bowling_seasons.csv  ({len(bowling_df)} rows)")

# Innings-level (for matchup matrix)
if 'venue' in economy_inn.columns:
    venue_bowl.to_csv("outputs/venue_bowling_profiles.csv", index=False)
if 'opponent' in fours_inn.columns:
    opp_bat.to_csv("outputs/opponent_batting_profiles.csv", index=False)

print("\nAll exports complete.")
print("\nMaster dataset columns:")
print(master.columns.tolist())

Saved: outputs/master_players.csv (652 rows, 35 cols)
Saved: outputs/batting_seasons.csv  (2148 rows)
Saved: outputs/bowling_seasons.csv  (1631 rows)

All exports complete.

Master dataset columns:
['player', 'total_runs', 'bat_avg', 'bat_sr', 'total_fours', 'total_sixes', 'centuries', 'fifties', 'seasons_batted', 'last_season_bat', 'total_wickets', 'bowl_avg', 'economy', 'bowl_sr', 'seasons_bowled', 'last_season_bowl', 'best_economy', 'avg_dots_per_spell', 'inn_wickets', 'max_sr_inn', 'avg_sr_inn', 'max_6s_inn', 'max_4s_inn', 'role', 'bat_impact', 'bowl_impact', 'overall_impact', 'powerplay_bowler', 'death_bowler', 'finisher', 'anchor', 'runs_3yr_avg', 'sr_3yr_avg', 'wkts_3yr_avg', 'econ_3yr_avg']


## 11. Quick EDA

In [16]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('IPL Dataset — Exploratory Analysis', fontsize=14, fontweight='bold')

# 1. Role distribution
role_counts = master['role'].value_counts()
axes[0,0].bar(role_counts.index, role_counts.values, color=['#00d4ff','#ff6b35','#00ff9d'])
axes[0,0].set_title('Player Role Distribution')
axes[0,0].set_ylabel('Count')

# 2. Top 10 batters by overall runs
top_bat = master.nlargest(10,'total_runs')[['player','total_runs']]
axes[0,1].barh(top_bat['player'], top_bat['total_runs'], color='#00d4ff')
axes[0,1].set_title('Top 10 Run Scorers')
axes[0,1].invert_yaxis()

# 3. Top 10 bowlers by wickets
top_bowl = master.nlargest(10,'total_wickets')[['player','total_wickets']]
axes[0,2].barh(top_bowl['player'], top_bowl['total_wickets'], color='#ff6b35')
axes[0,2].set_title('Top 10 Wicket Takers')
axes[0,2].invert_yaxis()

# 4. Bat SR distribution
bat_sr_data = master[master['bat_sr'] > 0]['bat_sr']
axes[1,0].hist(bat_sr_data, bins=30, color='#00d4ff', edgecolor='black', alpha=0.7)
axes[1,0].set_title('Batting Strike Rate Distribution')
axes[1,0].set_xlabel('Strike Rate')

# 5. Economy distribution
econ_data = master[(master['economy'] > 0) & (master['economy'] < 15)]['economy']
axes[1,1].hist(econ_data, bins=30, color='#ff6b35', edgecolor='black', alpha=0.7)
axes[1,1].set_title('Bowling Economy Distribution')
axes[1,1].set_xlabel('Economy Rate')

# 6. Impact score by role
for role, color in [('batter','#00d4ff'),('bowler','#ff6b35'),('allrounder','#00ff9d')]:
    data = master[master['role']==role]['overall_impact']
    axes[1,2].hist(data, bins=20, alpha=0.6, label=role, color=color)
axes[1,2].set_title('Overall Impact Score by Role')
axes[1,2].set_xlabel('Impact Score')
axes[1,2].legend()

plt.tight_layout()
plt.savefig("outputs/eda_overview.png", dpi=120, bbox_inches='tight')
plt.show()
print("EDA saved to outputs/eda_overview.png")

EDA saved to outputs/eda_overview.png


## Summary
| Output File | Description |
|---|---|
| `outputs/master_players.csv` | Full player profile with all features & impact scores |
| `outputs/batting_seasons.csv` | Season-level batting data (2008–2022) |
| `outputs/bowling_seasons.csv` | Season-level bowling data (2008–2022) |
| `outputs/venue_bowling_profiles.csv` | Bowler economy by venue |
| `outputs/opponent_batting_profiles.csv` | Batter SR by opponent |

**Next:** Load `master_players.csv` into `02_ML_Models.ipynb` for Win Probability & Recommender training.